# VoxGuard experiments in Google Colab

Compare the MFCC baseline, noise-augmented MFCC/LFCC classifiers, and a log-Mel CNN on real ASVspoof 2019 LA speech.

1. Choose **Runtime → Change runtime type → GPU** before running the CNN.
2. Run the cells in order. Training and development choose settings; the last experiment scores the untouched evaluation subset once.
3. Results and checkpoints are saved to your Google Drive by default. Audio and spectrogram caches stay on the temporary Colab disk.

The original pilot and the subsequent local comparison already inspected 1,400 evaluation utterances. This notebook excludes their IDs and hashes. Dataset attribution: [ASVspoof 2019](https://doi.org/10.7488/ds/2555), Open Data Commons Attribution License. No synthetic speech is generated; this project detects it.

The CNN has passed local software and one-epoch training checks. A complete GPU training experiment is still to be run. Better performance is an experimental objective, not a guarantee.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

PROJECT = Path('/content/voxguard')
REPOSITORY = 'https://github.com/asarekwaku/voxguard.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(PROJECT)], check=True)
elif not (PROJECT / '.git').exists():
    raise RuntimeError('The project folder exists but is not a Git checkout. Choose a fresh runtime.')
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-neural.txt'], check=True)
import torch
print('PyTorch:', torch.__version__, 'GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## Configure the run

The default experiment uses 6,000 training, 1,200 development, and 2,000 evaluation recordings. Every partition is balanced by class and approximately balanced by synthetic attack. For a quick functional run, lower the counts to 1,200 / 400 / 400 and epochs to 3. A short run does not establish CNN performance.

The download is selective and resumable by file. Download size and training time depend on the counts and the assigned hardware. Keep the evaluation seed fixed until the run is complete; do not use evaluation scores to change model settings.


In [ ]:
from datetime import datetime, timezone

TRAIN_COUNT = 6000
DEV_COUNT = 1200
EVAL_COUNT = 2000
SEED = 2475
EPOCHS = 25
BATCH_SIZE = 32
AUGMENTED_VIEWS = 1
USE_DRIVE = True
RUN_CNN = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_ROOT = Path('/content/drive/MyDrive/VoxGuard')
else:
    RESULTS_ROOT = Path('/content/voxguard_results')

# To resume an interrupted CNN run, set RUN_NAME to the existing run folder name.
RUN_NAME = datetime.now(timezone.utc).strftime('experiment_%Y%m%d_%H%M%S')
RUN = RESULTS_ROOT / RUN_NAME
DATA = Path('/content/voxguard_data')
CACHE = Path('/content/voxguard_cache')
RUN.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = '/content/matplotlib_config'
os.environ['XDG_CACHE_HOME'] = '/content/cache'

def run_command(*arguments):
    command = [sys.executable, *map(str, arguments)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=PROJECT, check=True)

print('Results:', RUN)


## Run software tests

These tests check the implementation. Speech-detection performance is measured later on the real recordings.

In [ ]:
run_command('-m', 'pytest', '-q')


## Download real speech with fresh evaluation IDs

The official training/development partitions stay separate. All locally inspected evaluation recordings are excluded. A record of exact IDs, hashes, and source archive members is saved with your run.

In [ ]:
run_command('scripts/download_subset.py', '--out', DATA,
            '--train', TRAIN_COUNT, '--dev', DEV_COUNT, '--eval', EVAL_COUNT,
            '--seed', SEED, '--workers', 8,
            '--exclude-manifest', 'protocols/inspected_evaluation.csv')
import shutil
shutil.copy2(DATA / 'manifest.csv', RUN / 'manifest.csv')
shutil.copy2(DATA / 'provenance.json', RUN / 'dataset_provenance.json')


## Train a comparable MFCC baseline

Retraining the baseline in Colab avoids loading scikit-learn models saved under a different library version. The same training and development recordings are used for the classical improvements and CNN. This baseline selects its settings using clean development recordings.


In [ ]:
BASELINE = RUN / 'baseline'
if not (BASELINE / 'selection.json').exists():
    run_command('-m', 'voxguard.cli', 'train', '--manifest', DATA / 'manifest.csv',
                '--out', BASELINE, '--seed', SEED, '--cache', CACHE / 'baseline_features')
else:
    print('Using the already frozen baseline in this run.')


## Train the improved acoustic classifiers

The combined representation includes MFCC and LFCC statistics. Each source clip contributes a clean view and independently seeded noise-augmented views. Hyperparameters and the score threshold are selected only from the clean and noisy development conditions.


In [ ]:
ACOUSTIC = RUN / 'acoustic'
if not (ACOUSTIC / 'frozen.json').exists():
    run_command('-m', 'voxguard.robust_train', '--manifest', DATA / 'manifest.csv',
                '--out', ACOUSTIC, '--seed', SEED, '--views', AUGMENTED_VIEWS,
                '--representations', 'combined')
else:
    print('Using the already frozen acoustic model in this run.')


## Train the CNN on the GPU

The CNN uses 64-band log-Mel spectrograms, training-only normalization, noise augmentation, and training-only frequency/time masks. A length mask limits pooling over padded time frames. The best epoch is selected using average development balanced accuracy across clean, 20 dB, and 10 dB conditions. Early stopping limits overtraining.

`best.pt`, `last.pt`, and `history.json` are saved to Drive after epochs. For a disconnected session, reuse the same RUN_NAME and data settings, rerun setup/download cells, and rerun this cell. It resumes an interrupted checkpoint. CUDA operations may not be bit-for-bit deterministic across hardware.


In [ ]:
CNN = RUN / 'cnn'
if RUN_CNN:
    if not torch.cuda.is_available():
        raise RuntimeError('Choose a GPU runtime, or set RUN_CNN=False to run only the acoustic experiments.')
    if not (CNN / 'frozen.json').exists():
        command = ['-m', 'voxguard.neural', '--manifest', DATA / 'manifest.csv',
                   '--out', CNN, '--cache', CACHE, '--epochs', EPOCHS,
                   '--batch-size', BATCH_SIZE, '--workers', 2,
                   '--views', AUGMENTED_VIEWS, '--seed', SEED, '--device', 'cuda']
        if (CNN / 'last.pt').exists():
            command.append('--resume')
        run_command(*command)
    else:
        print('Using the already frozen CNN in this run.')


## Evaluate once after all choices are frozen

This compares the clean-trained MFCC SVM, selected augmented acoustic model, and (if trained) CNN on exactly the same held-out recordings and noise. The MFCC SVM is the prespecified classical reference; its hyperparameters are selected by the baseline development run. These are subset results, not full-benchmark results.

Do not tune using these results. If you start a subsequent experiment, add this run's evaluation IDs to the exclusion list and reserve new data.


In [ ]:
BENCHMARK = RUN / 'fresh_benchmark'
command = ['-m', 'voxguard.fresh_benchmark', '--manifest', DATA / 'manifest.csv',
           '--exclude-manifest', 'protocols/inspected_evaluation.csv',
           '--baseline', BASELINE / 'svm_rbf.joblib',
           '--robust', ACOUSTIC / 'selected.joblib', '--out', BENCHMARK, '--seed', SEED]
if RUN_CNN:
    command += ['--cnn', CNN / 'best.pt', '--device', 'cuda']
if not (BENCHMARK / 'results.json').exists():
    run_command(*command)
else:
    print('Loading the completed benchmark without rescoring.')


## View and export results

The summary, per-recording predictions, frozen settings, model weights, and training history can be used in the milestone. Caches are excluded from the export. Report the actual scores, including failures and tradeoffs.

In [ ]:
import csv
import json
import zipfile
from IPython.display import display
import pandas as pd

results = json.loads((BENCHMARK / 'results.json').read_text())
summary = []
for model, conditions in results.items():
    for condition, metric in conditions.items():
        summary.append({'model': model, 'condition': condition,
                        'accuracy_percent': 100 * metric['accuracy'],
                        'synthetic_f1': metric['synthetic_f1'], 'roc_auc': metric['roc_auc'],
                        'eer_percent': 100 * metric['eer'],
                        'genuine_false_alarm_percent': 100 * metric['genuine_false_alarm_rate']})
frame = pd.DataFrame(summary)
display(frame.round(4))
frame.to_csv(RUN / 'summary.csv', index=False)

excluded = list(csv.DictReader(open(PROJECT / 'protocols/inspected_evaluation.csv')))
excluded += [row for row in csv.DictReader(open(DATA / 'manifest.csv')) if row['split'] == 'eval']
fields = ['split', 'utterance_id', 'speaker_id', 'sha256']
with (RUN / 'inspected_evaluation_next_run.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=fields, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(excluded)

archive_path = RUN.parent / (RUN.name + '.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in RUN.rglob('*'):
        if path.is_file() and 'feature_cache' not in path.parts:
            archive.write(path, path.relative_to(RUN))
print('Results saved:', RUN)
print('Export:', archive_path)
